# Code for data-curation for paper/thesis XXX

## Previous steps performed: 
1. All of PDB updated on the 2nd January 2026
2. All of PDB clustered with Foldseek, cut-off: 0.5, TM-align mode 
3. Split centerpoints into two files (16th Feb 2026)        
    a. Centerpoints before cutoff (30th September 2021)              
    b. Centerpoints where all entries are after cut-off (30th September 2021)                  
4. Classify all centerpoints stoichiometry


In [1]:
import pandas as pd
import numpy as np
import gzip
from Bio.PDB import MMCIFParser
from collections import defaultdict
from pathlib import Path
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

In [2]:
# Load data
post_cut = pd.read_csv("centers_cutoff_only.txt.classified.csv")
pre_cut = pd.read_csv("centers_all_excluding_cutoff.txt.classified.csv")

In [3]:
pd.set_option("display.max_rows", None, "display.max_columns", None)


In [4]:
# retrieve only data we're interested in
post_cut_filt = post_cut[post_cut["category"].isin(["homodimer","monomer","heterodimer"])]
pre_cut_filt = pre_cut[pre_cut["category"].isin(["homodimer","monomer","heterodimer"])]

In [6]:
# exclude data where different bio-assemblies have different stoichiometries
def remove_inconsistent_stoichiometry(df):
    # Extract base PDB id (before first "-")
    df = df.copy()
    df["pdb_id"] = df["centerpoint"].str.split("-", n=1).str[0]

    # Keep only PDB IDs where all assemblies have the same category
    consistent = (
        df.groupby("pdb_id")["category"]
        .nunique()
        .eq(1)
    )

    consistent_ids = consistent[consistent].index

    return df[df["pdb_id"].isin(consistent_ids)].drop(columns="pdb_id")


post_cut_clean = remove_inconsistent_stoichiometry(post_cut_filt)
pre_cut_clean  = remove_inconsistent_stoichiometry(pre_cut_filt)

In [7]:
# How many got removed? 
print("Post-cut removed:",
      len(post_cut_filt) - len(post_cut_clean))

print("Pre-cut removed:",
      len(pre_cut_filt) - len(pre_cut_clean))

Post-cut removed: 4
Pre-cut removed: 150


In [8]:
post_cut_clean.head(5)

,centerpoint,category
0,5sbg-assembly1,monomer
8,6xqj-assembly1,monomer
19,6zh1-assembly1_A,heterodimer
30,7acw-assembly1_A,heterodimer
31,7acw-assembly2_D,heterodimer


In [9]:
# remove proteins where any chain contains less than 30 amino acids
assembly_dir = Path("/mnt/sde/users/sarahn/entire_pdb")
parser = MMCIFParser(QUIET=True)

def assembly_has_short_chain(centerpoint, min_len=40):
    """
    Returns True if ANY chain in the structure is shorter than min_len.
    """
    cif_name = centerpoint.split("_")[0]
    cif_name = f"{cif_name}.cif.gz"
    cif_path = assembly_dir / cif_name

    if not cif_path.exists():
        print("Missing:", cif_path)
        return True  # choose to drop if missing

    with gzip.open(cif_path, "rt") as fh:
        d = MMCIF2Dict(fh)

    # Prefer explicit lengths if present
    lengths = d.get("_entity_poly.pdbx_seq_one_letter_code_can_length")
    if lengths:
        # Can be a single string or list of strings
        if isinstance(lengths, str):
            lengths = [lengths]
        lens = []
        for x in lengths:
            try:
                lens.append(int(x))
            except ValueError:
                pass
        if lens:
            return min(lens) < min_len

    # Fallback: compute lengths from sequences
    seqs = d.get("_entity_poly.pdbx_seq_one_letter_code_can") or d.get("_entity_poly.pdbx_seq_one_letter_code")
    types = d.get("_entity_poly.type")  # e.g. 'polypeptide(L)' / 'polydeoxyribonucleotide' etc.

    if not seqs:
        return True  # nothing polymeric -> not interesting for you

    if isinstance(seqs, str):
        seqs = [seqs]
    if isinstance(types, str):
        types = [types]

    def clean_len(seq: str) -> int:
        # remove whitespace/newlines; keep letters only
        return len("".join(seq.split()))

    # If types exists, restrict to proteins
    if types and len(types) == len(seqs):
        prot_lens = [clean_len(s) for s, t in zip(seqs, types) if "polypeptide" in (t or "").lower()]
    else:
        # If no types, assume these are polymer sequences and measure all
        prot_lens = [clean_len(s) for s in seqs]

    if not prot_lens:
        return True  # no protein polymers

    return min(prot_lens) < min_len


def remove_short_chains(df, min_len=40):
    keep = []
    for cp in df["centerpoint"]:
        keep.append(not assembly_has_short_chain(cp, min_len=min_len))
    return df[keep].reset_index(drop=True)

post_cut_final = remove_short_chains(post_cut_clean, 30)
pre_cut_final  = remove_short_chains(pre_cut_clean, 30)



In [10]:
# pre_cut_final.to_csv("monomers_homodimers_heterodimers_before_cutoff_above_30.csv")
# post_cut_final.to_csv("monomers_homodimers_heterodimers_after_cutoff_above_30.csv")

In [11]:
# how many removed? 
print("Post-cut removed:",
      len(post_cut_clean) - len(post_cut_final))

print("Pre-cut removed:",
      len(pre_cut_clean) - len(pre_cut_final))

Post-cut removed: 1707
Pre-cut removed: 7543


In [12]:
# print(len(pre_cut_final))
# print(len(post_cut_final[post_cut_final["category"].isin(["monomer"])]))
# print(len(pre_cut_final[pre_cut_final["category"].isin(["monomer"])]))
pre_cut_final[pre_cut_final["category"].isin(["monomer"])]["centerpoint"].head(10)

1       155c-assembly1
2       16vp-assembly1
3       1914-assembly1
6       1a0i-assembly1
7       1a0p-assembly1
8     1a1t-assembly1_A
10      1a1w-assembly1
11      1a2o-assembly2
13      1a41-assembly1
15      1a5j-assembly1
Name: centerpoint, dtype: str

In [13]:
# Filter away monomers that have clusters with members that are not monomers 
mmseqs_clusters = pd.read_table(
    "cluster-entire-pdb-09_cluster.with_stoich.tsv",
    header=None,
    names=["center", "member", "member_stoich"]
)

# normalize
mmseqs_clusters["member_norm"] = (
    mmseqs_clusters["member"]
    .astype(str)
    .str.strip()
    .str.lower()
)

mmseqs_clusters["member_stoich"] = (
    mmseqs_clusters["member_stoich"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Extract PDB id only (remove chain)
mmseqs_clusters["pdb_id"] = mmseqs_clusters["member_norm"].str.split("_").str[0]

# 1️⃣ Identify clusters that contain ANY non-monomer
bad_cluster_centers = (
    mmseqs_clusters.groupby("center")["member_stoich"]
    .apply(lambda s: (s != "monomer").any())
)

bad_cluster_centers = set(bad_cluster_centers[bad_cluster_centers].index)

# 2️⃣ Find monomer members (PDB IDs) inside those mixed clusters
bad_monomer_pdbs = set(
    mmseqs_clusters.loc[
        (mmseqs_clusters["center"].isin(bad_cluster_centers)) &
        (mmseqs_clusters["member_stoich"] == "monomer"),
        "pdb_id"
    ].unique()
)

# Function to extract pdb_id from your dataframe centerpoints
def df_to_pdb_id(cp):
    if pd.isna(cp):
        return None
    cp = str(cp).strip().lower()
    return cp.split("-")[0]  # remove -assembly1

def drop_monomers_in_mixed_clusters(df):
    df = df.copy()
    df["pdb_id"] = df["centerpoint"].map(df_to_pdb_id)

    drop_mask = (
        (df["category"] == "monomer") &
        (df["pdb_id"].isin(bad_monomer_pdbs))
    )

    return df.loc[~drop_mask].drop(columns=["pdb_id"]).reset_index(drop=True)

pre_cut_final2  = drop_monomers_in_mixed_clusters(pre_cut_final)
post_cut_final2 = drop_monomers_in_mixed_clusters(post_cut_final)

In [14]:
# How many got removed? 
print("Removed monomers (pre):",
      len(pre_cut_final) - len(pre_cut_final2))
print("Removed monomers (post):",
      len(post_cut_final) - len(post_cut_final2))

Removed monomers (pre): 1875
Removed monomers (post): 198


In [15]:
# def print_category_counts(df, name):
#     print(f"\n{name}")
#     print("-" * len(name))
#     counts = df["category"].value_counts().sort_index()
#     print(counts)
#     print("\nTotal:", len(df))


# print_category_counts(pre_cut_final2,  "Pre-cut (final)")
# print_category_counts(post_cut_final2, "Post-cut (final)")

def print_category_counts_with_unique_ids(df, name):
    df = df.copy()
    
    # Extract pdb_id (remove -assembly...)
    df["pdb_id"] = df["centerpoint"].str.split("-", n=1).str[0].str.lower()
    
    print(f"\n{name}")
    print("-" * len(name))
    
    # Row counts
    row_counts = df["category"].value_counts().sort_index()
    print("Row counts:")
    print(row_counts)
    
    # Unique pdb_id counts per category
    unique_counts = (
        df.groupby("category")["pdb_id"]
        .nunique()
        .sort_index()
    )
    
    print("\nUnique PDB IDs per category:")
    print(unique_counts)
    
    print("\nTotal rows:", len(df))
    print("Total unique PDB IDs:", df["pdb_id"].nunique())

print_category_counts_with_unique_ids(pre_cut_final2,  "Pre-cut (final)")
print_category_counts_with_unique_ids(post_cut_final2, "Post-cut (final)")

# final 40 residues cut-off
# Pre-cut (final)
# ---------------
# category
# heterodimer    1652
# homodimer      2733
# monomer        3431
# Name: count, dtype: int64

# Total: 7816

# Post-cut (final)
# ----------------
# category
# heterodimer    382
# homodimer      450
# monomer        449
# Name: count, dtype: int64

# Total: 1281



# final 20 residues cut-off
# Pre-cut (final)
# ---------------
# category
# heterodimer    3089
# homodimer      2842
# monomer        4197
# Name: count, dtype: int64

# Total: 10128 - I think this is a good start, it's much larger than what I had before

# Post-cut (final)
# ----------------
# category
# heterodimer    684
# homodimer      462
# monomer        562
# Name: count, dtype: int64

# Total: 1708

# 30 residues:, I will go with this.  

# Pre-cut (final)
# ---------------
# Row counts:
# category
# heterodimer    1971
# homodimer      2807
# monomer        3746
# Name: count, dtype: int64

# Unique PDB IDs per category:
# category
# heterodimer    1568
# homodimer      2475
# monomer        3728
# Name: pdb_id, dtype: int64

# Total rows: 8524
# Total unique PDB IDs: 7771

# Post-cut (final)
# ----------------
# Row counts:
# category
# heterodimer    456
# homodimer      459
# monomer        497
# Name: count, dtype: int64

# Unique PDB IDs per category:
# category
# heterodimer    379
# homodimer      400
# monomer        491
# Name: pdb_id, dtype: int64

# Total rows: 1412
# Total unique PDB IDs: 1270


Pre-cut (final)
---------------
Row counts:
category
heterodimer    1971
homodimer      2807
monomer        3746
Name: count, dtype: int64

Unique PDB IDs per category:
category
heterodimer    1568
homodimer      2475
monomer        3728
Name: pdb_id, dtype: int64

Total rows: 8524
Total unique PDB IDs: 7771

Post-cut (final)
----------------
Row counts:
category
heterodimer    456
homodimer      459
monomer        497
Name: count, dtype: int64

Unique PDB IDs per category:
category
heterodimer    379
homodimer      400
monomer        491
Name: pdb_id, dtype: int64

Total rows: 1412
Total unique PDB IDs: 1270


In [16]:
# pre_cut_final2[pre_cut_final2["category"]=="heterodimer"]

In [17]:
# pre_cut_final2.to_csv("mono_homo_hetero_before_cutoff_above_30_filt_mono_160225.csv")
# post_cut_final2.to_csv("mono_homo_hetero_after_cutoff_above_30_filt_mono_160225.csv")

In [18]:

# remove heterodimers with homodimers,homomultimers

def df_to_pdb_id(cp):
    if pd.isna(cp):
        return None
    cp = str(cp).strip().lower()
    return cp.split("-", 1)[0]   # 12as-assembly1_B -> 12as


def build_bad_pdbs_for_target(mm_df: pd.DataFrame, allowed_set: set[str]) -> set[str]:
    """
    Returns PDB IDs that appear in clusters that contain ANY disallowed member_stoich.
    (We do this by: find "bad" centers, then collect their member pdb_ids.)
    """
    bad_centers = set(
        mm_df.groupby("center")["member_stoich"]
        .apply(lambda s: (~s.isin(allowed_set)).any())
        .loc[lambda x: x].index
    )

    bad_pdbs = set(mm_df.loc[mm_df["center"].isin(bad_centers), "pdb_id"].unique())
    return bad_pdbs


def drop_contaminated_by_cluster(df: pd.DataFrame, target_category: str, allowed_set: set[str], mm_df: pd.DataFrame):
    """
    Drop rows where df['category'] == target_category AND their PDB ID is found in a cluster
    that contains ANY disallowed stoichiometry (not in allowed_set).
    """
    df = df.copy()
    df["pdb_id"] = df["centerpoint"].map(df_to_pdb_id)

    bad_pdbs = build_bad_pdbs_for_target(mm_df, allowed_set)

    drop_mask = (df["category"] == target_category) & (df["pdb_id"].isin(bad_pdbs))
    return df.loc[~drop_mask].drop(columns=["pdb_id"]).reset_index(drop=True)


In [19]:
RULES = {
    # "monomer": {"monomer"},
    "homodimer": {"homodimer", "homomultimer"},
    "heterodimer": {"heterodimer", "heteromultimer"},
}

def apply_rules(df, rules, mm_df):
    out = df
    for target_cat, allowed in rules.items():
        out = drop_contaminated_by_cluster(out, target_cat, allowed, mm_df)
    return out

pre_cut_final3  = apply_rules(pre_cut_final2,  RULES, mmseqs_clusters)
post_cut_final3 = apply_rules(post_cut_final2, RULES, mmseqs_clusters)

def dropped_counts(before, after):
    return (before["category"].value_counts() - after["category"].value_counts()).fillna(0).astype(int)

print("Pre dropped:\n", dropped_counts(pre_cut_final2, pre_cut_final3))
print("Post dropped:\n", dropped_counts(post_cut_final2, post_cut_final3))


Pre dropped:
 category
monomer           0
homodimer       989
heterodimer    1452
Name: count, dtype: int64
Post dropped:
 category
monomer          0
homodimer      156
heterodimer    311
Name: count, dtype: int64


In [20]:
print_category_counts_with_unique_ids(pre_cut_final3,  "Pre-cut (final3)")
print_category_counts_with_unique_ids(post_cut_final3, "Post-cut (final3)")

# Pre-cut (final2)
# Before filtering heterodimers and homodimers
# ---------------
# Row counts:
# category
# heterodimer    1971
# homodimer      2807
# monomer        3746
# Name: count, dtype: int64

# Unique PDB IDs per category:
# category
# heterodimer    1568
# homodimer      2475
# monomer        3728
# Name: pdb_id, dtype: int64

# Total rows: 8524
# Total unique PDB IDs: 7771

# Post-cut (final)
# ----------------
# Row counts:
# category
# heterodimer    456
# homodimer      459
# monomer        497
# Name: count, dtype: int64

# Unique PDB IDs per category:
# category
# heterodimer    379
# homodimer      400
# monomer        491
# Name: pdb_id, dtype: int64

# Total rows: 1412
# Total unique PDB IDs: 1270

# Pre-cut (final3)
# ---------------
# Row counts:
# category
# heterodimer     519
# homodimer      1818
# monomer        3746
# Name: count, dtype: int64

# Unique PDB IDs per category:
# category
# heterodimer     406
# homodimer      1622
# monomer        3728
# Name: pdb_id, dtype: int64

# Total rows: 6083
# Total unique PDB IDs: 5756

# Post-cut (final)
# ----------------
# Row counts:
# category
# heterodimer    145
# homodimer      303
# monomer        497
# Name: count, dtype: int64

# Unique PDB IDs per category:
# category
# heterodimer    109
# homodimer      263
# monomer        491
# Name: pdb_id, dtype: int64

# Total rows: 945
# Total unique PDB IDs: 863


Pre-cut (final3)
----------------
Row counts:
category
heterodimer     519
homodimer      1818
monomer        3746
Name: count, dtype: int64

Unique PDB IDs per category:
category
heterodimer     406
homodimer      1622
monomer        3728
Name: pdb_id, dtype: int64

Total rows: 6083
Total unique PDB IDs: 5756

Post-cut (final3)
-----------------
Row counts:
category
heterodimer    145
homodimer      303
monomer        497
Name: count, dtype: int64

Unique PDB IDs per category:
category
heterodimer    109
homodimer      263
monomer        491
Name: pdb_id, dtype: int64

Total rows: 945
Total unique PDB IDs: 863


In [ ]:
# print_category_counts_with_unique_ids(pre_cut_final2,  "Pre-cut (final2)")
# print_category_counts_with_unique_ids(post_cut_final2, "Post-cut (final2)")

# summary = pd.concat(
#     [
#         pre_cut_final2["category"].value_counts(),
#         post_cut_final2["category"].value_counts()
#     ],
#     axis=1
# )

# summary.columns = ["Pre-cut", "Post-cut"]
# summary = summary.fillna(0).astype(int)

# print(summary)
# print("\nTotals:")
# print("Pre-cut:", len(pre_cut_final2))
# print("Post-cut:", len(post_cut_final2))



Pre-cut (final2)
----------------
Row counts:
category
heterodimer    1971
homodimer      2807
monomer        3746
Name: count, dtype: int64

Unique PDB IDs per category:
category
heterodimer    1568
homodimer      2475
monomer        3728
Name: pdb_id, dtype: int64

Total rows: 8524
Total unique PDB IDs: 7771

Post-cut (final2)
-----------------
Row counts:
category
heterodimer    456
homodimer      459
monomer        497
Name: count, dtype: int64

Unique PDB IDs per category:
category
heterodimer    379
homodimer      400
monomer        491
Name: pdb_id, dtype: int64

Total rows: 1412
Total unique PDB IDs: 1270


In [ ]:

# save final3 to csv

pre_cut_final3.to_csv("mono_homo_hetero_before_training_cutoff_above_30_filt_mono_310326_temp.csv", index=False)
post_cut_final3.to_csv("mono_homo_hetero_after_training_cutoff_above_30_filt_mono_310326_temp.csv", index=False)